# Namma Space — train a photoreal Gaussian splat of the hostel (Kaggle)

**Settings (right-hand panel → Session options) before running:**
- **Accelerator → GPU T4 x2** (one T4 is used)
- **Internet → On** (needs a phone-verified Kaggle account)

Then **Run All**. About 45–75 minutes. When it finishes, open the **Output** panel (right side, `/kaggle/working`)
and download **`hostel_splat.ply`** (≈ 60 MB). Upload it to Google Drive and share the link.

What it does: installs Nerfstudio 1.1.5 → downloads the prepared dataset (1,376 PoseCam frames with ARCore poses,
300k-point start cloud; no COLMAP) → trains `splatfacto` 30,000 steps in the hostel's own metric frame →
measures quality on 137 held-out photos → exports and slims the splat for download.

In [ ]:
# 1. Check the GPU. If this stops: Session options -> Accelerator -> GPU T4 x2, then Run All again.
import subprocess, os
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit('NO GPU in this session. Session options -> Accelerator -> GPU T4 x2, then Run All again.')
print('GPU:', gpu.stdout.strip())
WORK = '/tmp/namma'            # dataset + training (not shown in Output)
OUT = '/kaggle/working'        # only the final files go here
os.makedirs(WORK, exist_ok=True)

In [ ]:
# 2. Install Nerfstudio (pins gsplat 1.4.0). Dependency-conflict warnings during install are harmless.
!pip install -q nerfstudio==1.1.5
import torch
assert torch.cuda.is_available(), 'PyTorch cannot see the GPU - check step 1'
print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'GPU', torch.cuda.get_device_name(0))

# Compile and test gsplat's GPU rasteriser now (several minutes the first time).
import gsplat
means = torch.randn(100, 3, device='cuda') + torch.tensor([0, 0, 3.0], device='cuda')
quats = torch.nn.functional.normalize(torch.randn(100, 4, device='cuda'), dim=-1)
img, alpha, _ = gsplat.rasterization(means, quats, torch.full((100, 3), 0.05, device='cuda'),
    torch.ones(100, device='cuda'), torch.rand(100, 3, device='cuda'), torch.eye(4, device='cuda')[None],
    torch.tensor([[[300.0, 0, 160], [0, 300.0, 120], [0, 0, 1]]], device='cuda'), 320, 240)
print('gsplat GPU rasteriser OK:', tuple(img.shape))

In [ ]:
# 3. Download the prepared hostel dataset (~45 MB). If this fails: Session options -> Internet -> On.
!wget -q -O /tmp/namma/hostel.zip https://github.com/atharv-namdeo/3d-mapping/raw/main/datasets/hostel_posecam_nerfstudio.zip
!unzip -q -o /tmp/namma/hostel.zip -d /tmp/namma/
DATA = f'{WORK}/hostel_posecam_nerfstudio'
print(len(os.listdir(f'{DATA}/images')), 'frames')

In [ ]:
# 4. Train (≈ 30–60 min). The four nerfstudio-data flags keep PoseCam's metric, gravity-aligned frame,
#    so the result lines up with the hostel's map, labels and routes.
import glob
cmd = ['ns-train', 'splatfacto', '--data', DATA,
       '--output-dir', f'{WORK}/outputs', '--experiment-name', 'hostel',
       '--max-num-iterations', '30000', '--vis', 'tensorboard',
       '--viewer.quit-on-train-completion', 'True',
       'nerfstudio-data', '--orientation-method', 'none', '--center-method', 'none',
       '--auto-scale-poses', 'False', '--load-3D-points', 'True']
with open(f'{WORK}/train.log', 'w') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        log.write(line)
        if 'Step' in line or '%' in line or 'rror' in line:
            print(line.rstrip()[-160:])
    proc.wait()
ckpts = sorted(glob.glob(f'{WORK}/outputs/hostel/splatfacto/*/nerfstudio_models/*.ckpt'))
if proc.returncode != 0 or not ckpts:
    print(open(f'{WORK}/train.log').read()[-4000:])
    raise SystemExit('Training failed (see the log above). Send the last lines to the team.')
config = os.path.join(os.path.dirname(os.path.dirname(ckpts[-1])), 'config.yml')
print('Trained:', ckpts[-1])

In [ ]:
# 5. Quality on the 137 held-out photos (higher PSNR = sharper; > 22 dB is good for 640x480 phone frames)
import json
ev = subprocess.run(['ns-eval', '--load-config', config, '--output-path', f'{OUT}/eval.json'], capture_output=True, text=True)
if ev.returncode == 0:
    res = json.load(open(f'{OUT}/eval.json'))['results']
    print({k: round(v, 3) for k, v in res.items() if not k.endswith('_std')})
else:
    print('Evaluation failed (the splat can still be exported):', ev.stderr[-1500:])

In [ ]:
# 6. Export, then slim the file for download: keep position, base colour, opacity, size and rotation
#    (the web viewer does not use the view-dependent colour terms, which are ~75 % of the file).
import re, shutil, numpy as np
subprocess.run(['ns-export', 'gaussian-splat', '--load-config', config, '--output-dir', f'{WORK}/export'], check=True)

def slim_splat(src, dst):
    with open(src, 'rb') as fh:
        header = b''
        while not header.endswith(b'end_header\n'):
            header += fh.readline()
        text = header.decode()
        n = int(re.search(r'element vertex (\d+)', text).group(1))
        props = re.findall(r'property float (\w+)', text)
        data = np.frombuffer(fh.read(n * 4 * len(props)), '<f4').reshape(n, len(props))
    keep = ['x', 'y', 'z', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'opacity', 'scale_0', 'scale_1', 'scale_2',
            'rot_0', 'rot_1', 'rot_2', 'rot_3']
    cols = data[:, [props.index(k) for k in keep]]
    head = 'ply\nformat binary_little_endian 1.0\nelement vertex %d\n' % n
    head += ''.join(f'property float {k}\n' for k in keep) + 'end_header\n'
    with open(dst, 'wb') as fh:
        fh.write(head.encode())
        fh.write(np.ascontiguousarray(cols, '<f4').tobytes())
    return n

n = slim_splat(f'{WORK}/export/splat.ply', f'{OUT}/hostel_splat.ply')
shutil.copy(os.path.join(os.path.dirname(config), 'dataparser_transforms.json'), OUT)
print(f'{n:,} gaussians ->', round(os.path.getsize(f'{OUT}/hostel_splat.ply') / 1e6, 1), 'MB')
!ls -la /kaggle/working

### Done — download and share
Output panel → `/kaggle/working` → download **`hostel_splat.ply`** (also `eval.json` for the quality numbers).
Upload it to Google Drive, share as *Anyone with the link*, and send the link. It is imported with
`"splatSource"` in the hostel's `scene.json` and `python -m navkit build app/scenes/hostel`; labels, search and
routes stay unchanged.